In [1]:
from platform import python_version
print(python_version())

3.11.14


### Cluster with Tahoe or sc-GTP

### Huggingface: tahoebio/Tahoe-x1-embeddings

https://github.com/tahoebio/tahoe-x1

Tahoe-x1: Scaling Perturbation-Trained Single-Cell Foundation Models to 3 Billion Parameters


#### Memory

That's not a general "64 GB isn't enough" — swap is fully exhausted at 2.0G, which means something asked for tens of GB in one allocation. Given where you are in the pipeline, the culprit is almost certainly load_tahoe_de, and the arithmetic says so:

The DE table is ~4.09e9 rows over ~75k conditions × ~54k genes. 

Filtering to pancreas doesn't help much — roughly 
- 6 lines × 379 drugs × ~4 doses × 54k genes ≈ 5e8 rows, 
- materialised in pandas with gene/drug/cell_line_id as object-dtype strings (~200 B/row) before pivot_table ever runs. 
- That's >100 GB. full_Z and consensus_cluster are megabytes by comparison.



In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [8]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



(PosixPath('/home/flavio/uv/perturb_agent/data/single_cell'), True)

### Running prism

In [9]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

Loaded /home/flavio/uv/perturb_agent/data/single_cell/deconv.h5ad (6.8 MB)
1604


In [10]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

In [11]:
res.theta_type.columns

Index(['Acinar cell', 'B cell', 'Ductal cell type 1', 'Endocrine cell', 'Endothelial cell',
       'Fibroblast cell', 'Macrophage cell', 'Stellate cell', 'T cell', 'malignant'],
      dtype='object')

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [12]:
res.cell_type_expression("Ductal cell type 1").shape

(1604, 153)

In [13]:
res.cell_type_expression("Ductal cell type 2").shape

(1604, 153)

### 2. theta is now fixed -> expand Z to every gene

In [14]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

verbose=True
force=False

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        force=force, verbose=verbose)

force=False
verbose=True
fname = "count-matrix.txt"
fname_ad = fname.replace('.txt', '.h5ad')

adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

filename_ad = prism.root_singc / fname_ad
compression = "gzip"
# adata.write_h5ad(filename_ad, compression=compression)
print(f"AData saved as {filename_ad},  ({filename_ad.stat().st_size/1e6:.0f} MB), compressed with {compression}")


verbose=True
fname_celltype = "all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

ref, s2t = prism.pseudobulk_reference(adata)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file
Table opened ((27169, 153)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_matrix.tsv'
Table opened ((153, 4)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_metadata.tsv'
57,530 cells x 24,005 genes | obs: []
AData saved as /home/flavio/uv/perturb_agent/data/single_cell/count-matrix.h5ad,  (338 MB), compressed with gzip
all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 24

In [16]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref)
Zfull.shape

(153, 10, 16550)

In [17]:
dic = {}

for cell_state in res.states:
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z
    print(cell_state, Z.shape)


Fibroblast cell (16550, 153)
Stellate cell (16550, 153)
Macrophage cell (16550, 153)
Endothelial cell (16550, 153)
T cell (16550, 153)
B cell (16550, 153)
Ductal cell type 2 (16550, 153)
Endocrine cell (16550, 153)
Ductal cell type 1 (16550, 153)
Acinar cell (16550, 153)


In [19]:
i=0
key = list(dic.keys())[i]

print(key)
Z = dic[key]
Z.head(3)

Fibroblast cell


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
A1BG,0.655,0.442,0.668,0.269,0.491,0.438,1.043,0.810,0.949,0.738,...,NaN,0.772,0.328,NaN,NaN,NaN,0.152,NaN,0.265,NaN
A1BG-AS1,3.150,1.251,1.249,2.249,1.408,1.848,3.725,1.988,2.017,2.263,...,NaN,6.944,9.022,NaN,NaN,NaN,7.544,NaN,1.126,NaN
A1CF,3.463,1.384,2.188,0.135,1.305,0.083,1.176,2.169,1.218,3.142,...,NaN,1.033,3.148,NaN,NaN,NaN,10.323,NaN,1.701,NaN


### Ductal 2 - malignant

In [20]:
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")
Zmal.shape

(16550, 153)

In [23]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]
prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

In [24]:
for g in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"]:
    if g in gfull:
        print(g, prism.gene_compartment_share(Zfull, gfull, res, g).head(3).round(3).to_dict())

FAM83A-AS1 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
HOXA10-AS {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
HOXB-AS3 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
MIR7-3HG {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}


### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [25]:
[g for g in prog1 if g in df_bulk.index]

['FAM83A-AS1', 'HOXA10-AS', 'HOXB-AS3', 'HOXB-AS4', 'MIR7-3HG']

### present in the scRNA reference?

In [26]:
  
[g for g in prog1 if g in ref.columns]

['FAM83A-AS1', 'HOXA10-AS', 'HOXB-AS3', 'MIR7-3HG']

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [27]:
set(ref.index.to_list())

{'Acinar cell',
 'B cell',
 'Ductal cell type 1',
 'Ductal cell type 2',
 'Endocrine cell',
 'Endothelial cell',
 'Fibroblast cell',
 'Macrophage cell',
 'Stellate cell',
 'T cell'}

In [28]:
set(s2t.index.to_list())

{'Acinar cell',
 'B cell',
 'Ductal cell type 1',
 'Ductal cell type 2',
 'Endocrine cell',
 'Endothelial cell',
 'Fibroblast cell',
 'Macrophage cell',
 'Stellate cell',
 'T cell'}

In [29]:
adata.obs

,cluster,cell_type,cell_state
cell,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell
...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1


In [30]:
import re, numpy as np, pandas as pd

adata.obs["sample"] = adata.obs_names.to_series().str.extract(r"^([TN]\d+)_")[0].values
adata.obs["tissue"] = np.where(adata.obs["sample"].str.startswith("T"), "tumor", "normal")

print(adata.obs.groupby("tissue")["sample"].nunique())      # expect tumor 24, normal 11
print(pd.crosstab(adata.obs["cell_state"], adata.obs["tissue"]))

tissue
normal    11
tumor     24
Name: sample, dtype: int64
tissue              normal  tumor
cell_state                       
Acinar cell           1423    512
B cell                  31   2416
Ductal cell type 1    7671   2646
Ductal cell type 2       0  11315
Endocrine cell         270    459
Endothelial cell      3983   5134
Fibroblast cell        940   5802
Macrophage cell        559   4802
Stellate cell          623   5284
T cell                  44   3616


### Count Malignant Cells - accordingo to transcriptomics

In [31]:
d2 = adata.obs["cell_state"].eq("Ductal cell type 2")
print(len(d2))
d2[:5]

57530


cell
T1_AAACCTGAGATGTCGG    False
T1_AAACGGGGTCATGCAT    False
T1_AAAGATGCATGTTGAC    False
T1_AAAGATGGTCGAGTTT    False
T1_AAAGATGGTCTCTCTG    False
Name: cell_state, dtype: bool

In [32]:
is_t = adata.obs["tissue"].eq("tumor")
print(np.sum(is_t))

41986


In [33]:
adata.obs["cell_state"] = np.where(d2 &  is_t, "Malignant ductal",
                          np.where(d2 & ~is_t, "Ductal cell type 2 normal",
                                   adata.obs["cell_state"]))
adata.obs

,cluster,cell_type,cell_state,sample,tissue
cell,,,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell,T1,tumor
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell,T1,tumor
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell,T1,tumor
...,...,...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell,N11,normal
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell,N11,normal
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1,N11,normal


In [34]:
from collections import Counter

Counter(adata.obs["cell_state"] )

Counter({'Malignant ductal': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [35]:
adata.obs["cell_type"]  = np.where(adata.obs["cell_state"].eq("Malignant ductal"),
                                   "malignant", adata.obs["cell_type"])

Counter(adata.obs["cell_type"] )

Counter({'malignant': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [36]:
ref2, s2t = prism.pseudobulk_reference(adata, state_key="cell_state", type_key="cell_type")
s2t.to_dict()

{'Fibroblast cell': 'Fibroblast cell',
 'Stellate cell': 'Stellate cell',
 'Macrophage cell': 'Macrophage cell',
 'Endothelial cell': 'Endothelial cell',
 'T cell': 'T cell',
 'B cell': 'B cell',
 'Malignant ductal': 'malignant',
 'Endocrine cell': 'Endocrine cell',
 'Ductal cell type 1': 'Ductal cell type 1',
 'Acinar cell': 'Acinar cell'}

### LFC

calc_celltype_lfc() — each compartment vs the mean of the others, paired across samples by default. Paired is the right default here because every sample contributes every cell type, so pairing removes cohort/purity variance. This doubles as deconvolution QC: if the ductal compartment doesn't recover KRT19/TFF1/CEACAM6 and fibroblast doesn't recover COL1A1/POSTN, θ or the Peng reference is off and step 2 is meaningless.

### Critics

- Why not, for each cell type, tumor samples x normal samples
- Only Ductal 2 Tumor has no normal samples - to confirm


In [37]:
res.__dict__.keys()

dict_keys(['theta', 'theta_stage1', 'theta_type', 'tumor_purity', 'genes', 'Z', 'states'])

In [38]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

### Prism programs

In [39]:
Z_full, genes_full = prism.full_Z(res, df_bulk, ref)

### MalignantCluster

In [40]:
# del(MalignantCluster)

In [41]:
from libs.prism_malig_lib import MalignantCluster

In [42]:
type(res)

libs.prism_lib.DeconvResult

In [43]:
cbio.root_mprog_disease

PosixPath('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD')

In [60]:
root_mprog_cluster = cbio.root_mprog_disease / 'cluster'
mal_cell_name = "Ductal cell type 2"
kmax = 8
no_decouple = True
is_tahoe = True

'''
mc = pml.MalignantCluster(prism=prism, res=res, 
                          df_bulk=df_bulk, ref=ref, 
                          root_mprog_cluster = root_mprog_cluster,
                          mal_cell_name = mal_cell_name,
                          organ="Pancreas")
'''

import importlib, libs.prism_malig_lib as pml
importlib.reload(pml)
print(pml.__version__)

0.33.0


In [61]:
mc = pml.MalignantCluster(prism=prism, res=res, 
                          df_bulk=df_bulk, ref=ref, 
                          root_mprog_cluster = root_mprog_cluster,
                          mal_cell_name = mal_cell_name,
                          organ="Pancreas")

mc

In [62]:
X, diag = mc.prepare_malignant_matrix(decouple_purity=False, 
                                      keep_genes=mc.program1_panel, drop_pattern=r"^N-")

print("Limited by n_hvg\n")
print(X.shape)
X.head(3)

excluded 22/153 samples by keep_samples/drop_pattern
Limited by n_hvg

(117, 2000)


,A1CF,AACS,AADAC,AATK,ABAT,ABCA12,ABCA7,ABCB9,ABCC3,ABCC6,...,ZNF774,ZNF787,ZNF792,ZNF816,ZNF888,ZNRF1,ZNRF2,ZSCAN29,ZSWIM5,ZWINT
T-C3L-02890,5.982,6.258,5.175,5.251,5.758,4.525,6.916,5.286,8.488,3.813,...,3.765,5.004,5.415,5.472,6.948,4.979,5.810,6.161,3.973,5.391
T-C3L-03635,4.757,5.724,1.766,4.362,5.829,6.391,5.053,3.572,9.914,2.841,...,4.566,4.348,5.478,6.968,7.315,4.711,6.219,6.158,3.293,5.177
T-C3L-02701,5.355,6.172,2.614,5.341,5.771,4.948,6.295,3.261,9.049,3.479,...,4.707,4.696,4.896,5.616,6.857,4.948,5.456,6.081,4.002,5.462


In [54]:
lista = [x for x in X.index if x.startswith('T-')]
X.shape, len(lista) == X.shape[0]

((117, 2000), True)

In [55]:
diag.keys()

dict_keys(['samples_excluded_by_filter', 'samples_dropped', 'n_genes_expressed', 'n_genes_share_not_computable', 'n_genes_share_ok', 'forced_genes_status', 'n_genes_kept', 'n_hvg', 'pc_theta_pearson_raw', 'pc_theta_pearson', 'decouple_purity', 'pc_theta_note', 'sample_mean_expr', 'sample_total_Z', 'theta_mal', 'n_samples_used', 'theta_excluded', 'theta_kept'])

In [56]:
diag["samples_excluded_by_filter"]

['N-C3L-04072',
 'N-C3L-00589',
 'N-C3L-03123',
 'N-C3L-04080',
 'N-C3L-00640',
 'N-C3N-01719',
 'N-C3L-07033',
 'N-C3L-00819',
 'N-C3L-07032',
 'N-C3L-01689',
 'N-C3N-01899',
 'N-C3N-00517',
 'N-C3N-03069',
 'N-C3N-02765',
 'N-C3L-07037',
 'N-C3N-02589',
 'N-C3N-02996',
 'N-C3L-02606',
 'N-C3N-03173',
 'N-C3N-02696',
 'N-TCGA-H6-8124',
 'N-TCGA-H6-A45N']

### pc_theta_pearson and pc_theta_pearson_raw

**The computation.** Run PCA on the samples × genes matrix, take the first 5 principal components, and correlate each PC's sample scores with `theta_mal` (each sample's malignant fraction). `pcs[:, i]` is one number per sample for PC *i*; `theta_mal.values` is one number per sample. `np.corrcoef(...)[0,1]` pulls the off-diagonal — the Pearson r between them.

You get 5 numbers, one per PC. Each answers: *does this dominant axis of variation track tumour purity?*

**The two versions:**

| | matrix | meaning |
|---|---|---|
| `pc_theta_pearson_raw` | `logx` — log2-CPM before decoupling | how much purity is in the data |
| `pc_theta_pearson` | `Xc` — the matrix you actually cluster | how much purity survives into the analysis |

With `decouple_purity=False` they're the same matrix, so the numbers match — your `[-0.596, -0.205, 0.227, -0.359, -0.158]` versus `[-0.596, -0.205, 0.228, -0.359, -0.159]`. The tiny differences are HVG selection, which happens between the two calls.

With `decouple_purity=True`, `Xc` holds residuals from regressing on `theta_mal`, and residuals are **orthogonal to their regressors by construction**. So `pc_theta_pearson` becomes ~1e-15 — pure floating-point noise. It proves the arithmetic worked, nothing about your data. That's why 0.20.1 added the `_raw` version and the `pc_theta_note`: I originally had you reading a number that can only ever be zero.

**Your actual numbers matter.** PC1 at r = −0.596 means ~36% of the leading component's variance is shared with purity, and PC4 at −0.359 adds more. The sign says low-purity samples score high on PC1. Since `X` is what produced the consensus clustering, the 134-gene tumour axis, and the 6/119 splits, purity is a live confound in all of them.

Which is the concrete reason to run `decouple_purity=True` and compare — with the standing caveat that basal-like PDAC is genuinely lower-purity, so some of that r is biology you'd be deleting.

In [ ]:
diag["pc_theta_pearson"]   # PC-vs-theta_mal

[-0.5957714654837519,
 -0.2052535664738762,
 0.2271564928280671,
 -0.3591899212713359,
 -0.1582042676862124]

In [ ]:
diag["pc_theta_pearson_raw"]   # PC-vs-theta_mal on logx (pre-decoupling)

[-0.5958663649566321,
 -0.20488699250477516,
 0.22757736818819427,
 -0.35926294739602765,
 -0.15958438869784913]

In [59]:
diag["pc_theta_note"]          # warns the decoupled version is ~0 by construction

'decouple_purity=False, so pc_theta_pearson and pc_theta_pearson_raw are the same matrix and both are informative: a large |r| on an early PC means the clustering is tracking tumour purity.'

In [ ]:
diag["sample_mean_expr"]       # Xc.mean(axis=1) per sample

In [ ]:
diag["sample_total_Z"]         # ms.Z.sum(axis=1) per sample

In [ ]:
diag["theta_excluded"]

In [ ]:
diag["theta_kept"]

### Inspecting vars

In [ ]:
import inspect
print(pml.__version__)
print("drop_pattern" in inspect.signature(mc.prepare_malignant_matrix).parameters)

In [ ]:
info = mc.inspect_de_schema(genes=X.columns)
print(info.keys())
info["columns"]

In [ ]:
info["matches"]

In [ ]:
info["dtypes"]

In [ ]:
info["resolved_columns"]

In [ ]:
info["numeric_profile"]      # min / max / mean / frac_negative / n_unique


In [ ]:
info["signed_candidates"]

In [ ]:
cov = mc.index_coverage()
cov["n_lines_seen"], cov["n_lines_in_metadata"]

In [ ]:
cov["block_probe_counts"]        # min probes per block

In [ ]:
lista = cov["in_metadata_not_in_index"]
len(lista), lista[:5]

In [ ]:
cov["n_lines_seen"], cov["n_lines_in_metadata"]

That bears directly on the result you just got. X_fib2 has 9959 genes, so the large majority are recovered rather than fitted. If those 367 stroma-axis hits are concentrated among recovered genes, the signal may be a property of the projection, not the fibroblast compartment.

It's a decomposition — a way of writing one number as a sum of two unobserved parts.

`z_mal[s,g]` is what you have: the z-scored malignant-compartment expression of gene *g* in sample *s*. One number per cell of `X_mal2`. The claim is that this number reflects two things at once:

- **`B[s,g]`** — the part shared with the fibroblast compartment. Both compartment matrices are reconstructions from the same bulk profile, so whatever made gene *g* high in sample *s*'s bulk pushes both up together.
- **`M[s,g]`** — the part specific to the malignant compartment: what the deconvolution actually recovered about tumour cells in that sample.

You never observe `B` or `M` separately. But you can measure their relative size, and that's what the correlation did:

```
z_mal = B + M
z_fib = B + F        (F = fibroblast-specific part)
```

If `M` and `F` are independent of each other and of `B`, then

```
corr(z_mal, z_fib) = var(B) / (var(B) + var(M))     ≈ 0.93
```

which is your r = 0.927. Hence "shared dominates roughly 13-to-1" — `var(B)` is about 13× `var(M)`.

You can also see it directly:

```python
B_hat = (zm[g] + zf[g]) / 2      # averaging cancels the specific parts
M_hat = (zm[g] - zf[g]) / 2      # differencing cancels the shared part
B_hat.var().median(), M_hat.var().median()
```

That's the same trick as the program contrast, one level down: **summing isolates what's shared, differencing isolates what's specific.** `basal − classical` does it across genes within a compartment; `z_mal − z_fib` does it across compartments for one gene.

In [63]:
cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
}

X_mal2 = mc.compartment_matrix(cmap["malignant"],  min_share=0.3, min_counts=10)
X_fib2 = mc.compartment_matrix(cmap["fibroblast"], min_share=0.3, min_counts=10)
print(X_mal2.shape, X_fib2.shape)

scores, cov = mc.program_scores(compartment_map=cmap, samples=X.index)   # tumours only

disc = mc.discretize_axes(scores)

print("in malignant", end=" ")
R_mal = mc.factorial_state_de(X_mal2, disc)
print("in fibroblast", end=" ")
R_fib = mc.factorial_state_de(X_fib2, disc)

print("-----------"*5)

fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)   # the 1604
hits = R_fib.index[R_fib.filter(like="fdr_B_").iloc[:,0] < 0.05]
print(f"{len(set(hits) & fitted)}/{len(hits)} hits are fitted genes")
print(f"background: {len(fitted & set(R_fib.index))}/{len(R_fib)}")

print("-----------"*5)

common_genes = X_mal2.columns.intersection(X_fib2.columns)
Z_mal_common = ((X_mal2[common_genes] - X_mal2[common_genes].mean()) / X_mal2[common_genes].std())
Z_fib_common = ((X_fib2[common_genes] - X_fib2[common_genes].mean()) / X_fib2[common_genes].std())
Z_corr = pd.Series({gene: Z_mal_common[gene].corr(Z_fib_common[gene]) for gene in common_genes})

fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)
print("fitted    genes, median r:", Z_corr[ Z_corr.index.isin(fitted)].median().round(3))
print("projected genes, median r:", Z_corr[~Z_corr.index.isin(fitted)].median().round(3))

(131, 8699) (130, 10014)
in malignant excluded 47 program marker genes; 8652 remain
in fibroblast excluded 55 program marker genes; 9959 remain
-------------------------------------------------------
109/367 hits are fitted genes
background: 558/9959
-------------------------------------------------------
fitted    genes, median r: 0.927
projected genes, median r: 0.905


### Opposite programs

Three hardcoded pairings that add a signed axis column whenever both poles were successfully scored.

**Why these three pairs.** Each is an opposed pair in the literature — a phenotype where high on one pole means low on the other. 
- Moffitt's basal-like/classical, 
- Öhlund's myCAF/iCAF, and 
- T-cell cytotoxic/exhausted. 

**Not every program has a natural opposite (`prolif`, `emt`, `TAM` don't)**, which is why only three are defined.

**Why the guards.** `program_scores` skips any program with fewer than `min_genes` markers found, so `scores` may not contain both poles. The `in` checks avoid a `KeyError` — and this is why your run has `fibroblast.axis_myCAF_minus_iCAF` but no immune axis: the T-cell programs didn't survive marker coverage at that compartment's θ.

**Why the naming matters.** The prefix is the *compartment* (`malignant.`, `fibroblast.`), not `axis.`. That was a bug I fixed earlier: with an `axis.` prefix, `couple_compartments` parses the compartment as `"axis"` and treats every axis as its own compartment — so it would correlate `malignant.axis_basal_minus_classical` against `malignant.basal`, which is trivially r≈0.99 since one contains the other. The current naming keeps each axis inside its compartment, and cross-compartment-only filtering excludes those self-comparisons.

**Why the subtraction is the point** — this is the part connecting to what we just discussed. A single pole score is ~93% shared bulk signal. The difference cancels it, because whatever inflates a sample's overall expression inflates both marker sets alike. So the axis is far closer to a compartment-specific measurement than either pole alone.

One caveat on the arithmetic: the two poles have different marker counts (basal 12, classical 12; myCAF 8, iCAF 9) and each is a mean of z-scores, so they're on comparable scales — but the difference is *not* itself standardised. Its variance depends on how correlated the two poles are. That's fine for correlation-based analyses (`couple_compartments`, `axis_modality`) since those are scale-invariant, but it means axis values aren't directly comparable in magnitude across compartments.

If you want to add pairs — `macrophage.M1` vs `macrophage.TAM` is the obvious candidate — the pattern is mechanical, though you'd want to be sure the opposition is real rather than assumed. M1/M2 polarisation in particular is increasingly regarded as a spectrum with co-expression, so a difference score there is a stronger assumption than for basal/classical.

In [67]:
scores.iloc[:10, -8:]

,macrophage.M1,macrophage.TAM,macrophage.SPP1,endothelial.tip_angio,endothelial.lymphatic,endothelial.activated,malignant.axis_basal_minus_classical,fibroblast.axis_myCAF_minus_iCAF
T-C3L-00277,NaN,NaN,NaN,0.947,-0.479,-1.527,-0.319,0.205
T-C3L-00589,-0.883,0.103,-0.497,-0.798,-0.485,-1.500,-0.212,-0.244
T-C3L-00625,NaN,NaN,NaN,1.001,0.298,-0.635,-0.434,-0.024
T-C3L-00640,0.083,-0.292,0.644,NaN,NaN,NaN,-0.703,-0.218
T-C3L-00819,NaN,NaN,NaN,NaN,NaN,NaN,0.294,0.378
T-C3L-00881,0.449,-0.188,-0.377,-0.258,-0.395,0.099,-0.083,-0.727
T-C3L-01032,NaN,NaN,NaN,NaN,NaN,NaN,1.713,0.182
T-C3L-01051,NaN,NaN,NaN,0.165,-0.163,-1.178,-1.193,0.915
T-C3L-01124,0.383,-0.386,-0.649,-0.394,-0.190,-0.247,-0.703,-0.385
T-C3L-01598,-0.611,-0.197,1.157,0.354,-0.172,-0.081,0.244,-0.042


### **The enrichment test is reassuring** and **The correlation test is the problem.**

**The enrichment test is reassuring.** 
- 109/367 hits are fitted genes (29.7%) 
  - against a background of 558/9959 (5.6%) 
  - a **5.3× enrichment**
  - not the depletion I warned about. 
  - so the 367-gene stromal signal concentrates in genes with genuine Gibbs posterior evidence, not in the projected majority. My concern there was wrong.

**The correlation test is the problem.** 
- I predicted projected genes near r=1 and fitted genes clearly lower. 
- Instead: fitted **0.927**, projected **0.905** — both extremely high, and fitted is *higher*.

**That means the malignant and fibroblast compartment matrices are ~92% correlated across samples, gene by gene, after z-scoring.**

Roughly 85% of the variance is shared. The deconvolution is separating compartment *means* (which is why marker programs score sensibly)
- but not compartment-specific *between-sample variation*. **Both matrices largely track the bulk.**

Note this is with `decouple=True` (the 0.31+ default), so it isn't θ — residualising on composition didn't remove it.

**Why your axis analyses survive this and the gene-level ones don't.** A program score is a contrast within a compartment: `basal − classical` subtracts the shared component, leaving the compartment-specific residual. That's why the axes behaved, why `couple_compartments` gave interpretable off-diagonal nulls, and why the `prolif`↔`iCAF` replication is credible. `factorial_state_de` on raw `X_mal2` vs `X_fib2` has no such cancellation — you're running DE on two near-copies of the same matrix.

So the 7-vs-367 contrast between compartments **cannot be read as compartment-specific biology**. Quantify the split directly:

If the specific fraction is ~0.04 for both classes, gene-level compartment claims aren't supportable at all and should be dropped from the writeup, keeping the contrast-based results.

This belongs in the summary as a limitation, and it's a genuine one: BayesPrism recovered compartment identity but not independent compartment-level sample variation. It also retroactively explains the zero off-diagonal in the factorial — with matrices this correlated, a "cross-compartment" test isn't really crossing anything.

### Shards - A database shard, or simply a shard, is a horizontal partition of data within a database or search engine.

In [ ]:
cvcl = mc.organ_cell_lines() 
mc.shard_index(stride=6)          # ~120 new footers, ~9 min (not 172)
shards = mc.find_shards_for(cvcl) # expect: located 11, missing 0

In [ ]:
len(cvcl), cvcl[:3]

In [ ]:
dropped = ('CVCL_0186','CVCL_1634','CVCL_1638','CVCL_1639')
cvcl7  = [c for c in cvcl if c not in dropped]
shards = mc.find_shards_for(cvcl7)          # expect missing: 0

In [ ]:
mc.shard_sizes(shards)

In [ ]:
mc.shard_sizes(shards)["bytes"].sum() / 1e9      # GB for the 237 shards

In [ ]:
# to much: lets paralelize
# mc.download_shards(shards, dry_run=True)
mc.download_shards(shards, max_workers=8)

In [ ]:
"""
df["condition"] = df["cell_line_id"].astype(str) + "|" + df["drug"].astype(str)
df_pivot = (df.pivot(index="gene", columns="condition", values="stat")
        .astype(dtype))
"""

df_pivot, cond = mc.load_tahoe_de(genes=X.columns.to_list(), organs=("Pancreas",),
                                  mode="download", _shard_subset=shards, force=True)

In [ ]:
print(df_pivot.shape)                                  # genes x conditions
df_pivot.head(8)

In [ ]:
cond.head(3)

In [ ]:
cond["cell_line_id"].nunique(), cond["drug"].nunique()

In [ ]:
cond["cell_line_id"].value_counts()     # expect 7 lines

In [ ]:
cond["drug"].value_counts()


In [ ]:
print("\n".join(np.unique(cond["drug"])))

In [ ]:
## 2D-values, stacked distribution
df_pivot.stack().describe()

In [ ]:
np.sum(np.sum(df_pivot<=1))

In [ ]:
np.sum(np.sum(df_pivot<=-1))

In [ ]:
float((df_pivot <= -1).mean().mean())

In [ ]:
(df_pivot >= 1).mean(axis=0)

In [ ]:
(df_pivot >= 1).mean(axis=1)

### Cluster

#### All three checks pass

- 0.4989 negative confirms stat is a genuinely signed statistic 
- the WTCS sign convention is sound. 
- 1986 of 2000 HVGs found in Tahoe is 99.3% coverage, better than I expected for a Parse 3' assay.

#### Score it:

In [ ]:
cc = mc.consensus_cluster(X)
cc

### PAC

Proportion of Ambiguous Clustering — a measure of how decisively the consensus clustering assigns samples, from Șenbabaoğlu et al. (2014). 

It's how choose_k picks k.

The consensus matrix C[i,j] is the fraction of resamples in which samples i and j landed in the same cluster, given both were drawn. 

Perfect structure gives values of 0 or 1 — pairs always together or always apart. Unstable structure gives values scattered in between.

PAC is just the fraction of pairs sitting in that ambiguous middle:



In [ ]:
consensus = cc[2]['consensus']
consensus.iloc[:5, :10]


In [ ]:
def _pac(consensus, lo=0.1, hi=0.9):
    v = consensus[np.triu_indices_from(consensus, k=1)]
    return float(((v > lo) & (v < hi)).mean())

_pac(consensus.values, lo=0.1, hi=0.9)

Low PAC = crisp, reproducible partition. choose_k takes the smallest k within pac_tol of the minimum, preferring parsimony when several k are comparably stable.

Why it misled you here. PAC rewards reproducibility, not biological meaning, and those come apart badly for unbalanced splits. Peeling six outliers off 119 samples is maximally reproducible — every resample isolates them identically — so PAC approaches zero. The metric is behaving exactly as designed while pointing at nothing interesting.

It's also mechanically biased toward small k, since fewer clusters means fewer boundaries to disagree about. That's why choose_k at k=2 deserves scepticism rather than confidence on its own.

So read the diagnostics together:

In [ ]:
dfa = pd.DataFrame({k: {"pac": v["pac"], "coph": v["cophenetic"],
                    "sil": v["silhouette"],
                    "sizes": v["labels"].value_counts().tolist()}
                    for k, v in cc.items()}).T

dfa

The sizes column is the one that would have caught this. 

A k with low PAC and balanced clusters is trustworthy; 
low PAC with a 6/119 split is an outlier detector. 

Cophenetic correlation and silhouette are worth glancing at too, though both share the same blind spot — none of them knows the difference between a real subtype and six weird samples.

In [ ]:
k = mc.choose_k(cc)
k

### k=2 is the expected answer for PDAC 

- Moffitt's classical vs basal-like is a two-group axis. 
- The earlier problem wasn't k=2, it was the 6-vs-119 split. 
- So the question now is what the two groups are.

Three checks, in order of how much they'd change your interpretation:

In [ ]:
labels = cc[2]["labels"]
print('n counts', labels.value_counts().to_dict())          # balanced now?
print("")
print(diag["pc_theta_pearson"])                 # PC1 vs theta_mal
print("")
print(pd.crosstab(labels, pd.Series(
    ['TCGA' if 'TCGA' in s else 'CPTAC' for s in X.index], index=X.index)))

A near-even split with |r| below ~0.3 on PC1 is what you want.

- If the crosstab shows the split tracking TCGA vs CPTAC, it's a batch axis — plausible given your strandedness history, 
- and it would mean the unstranded harmonisation didn't fully remove the cohort effect.

Then the test that actually names the clusters:

In [ ]:
basal = ["KRT81","KRT5","KRT6A","KRT17","S100A2","SPRR3","TP63","DHRS9","VGLL1"]
clas  = ["GATA6","TFF1","TFF2","TFF3","LGALS4","CLDN18","CEACAM6","AGR2", "ANXA10","REG4","CTSE","MUC13"]
Z = (X - X.mean()) / X.std()
sc = pd.DataFrame({
    "basal":     Z[[g for g in basal if g in X.columns]].mean(axis=1),
    "classical": Z[[g for g in clas  if g in X.columns]].mean(axis=1)})
print(sc.groupby(labels).mean().round(2))

If **one cluster is basal-high/classical-low** and **the other the reverse**, you've recovered Moffitt in the deconvolved malignant compartment

- which is a genuinely stronger result than the bulk clustering you started with, 
- because it's not confounded by stromal content. That was the whole point of the deconvolution detour.

If instead both clusters co-elevate the two programs, 
- you're seeing the same cellularity axis as before, and the purity decoupling didn't clear it.

Note how few of those markers likely survived your HVG filter — check [g for g in basal+clas if g in X.columns] first. If coverage is thin, score on the un-HVG-filtered logx instead, since marker scoring doesn't need the variance selection.

In [ ]:
labels = cc[k]["labels"]
sig    = mc.cluster_signatures(X, labels)
sig[1]

In [ ]:
"; ".join(sig[1]['stat'].index.to_list())

In [ ]:
"; ".join(sig[2]['stat'].index.to_list())

p = 4.8e-48 with n=6 is not credible. Welch t with ~5 df cannot produce that unless the within-group variance of the 6 is near zero. That's the fingerprint of prior domination: BayesPrism shrank all six toward the same reference profile, so they're nearly identical to each other. Tiny SE → exploding t → absurd p. Check directly:

In [ ]:
small = labels[labels==1].index
big   = labels[labels==2].index
pd.DataFrame({
    "sd_small": X.loc[small].std(axis=0),
    "sd_big":   X.loc[big].std(axis=0),
}).describe().round(3)

### Welch test

In [ ]:
labs1 = labels[labels==1].index
labs2 = labels[labels==2].index

X1 = X.loc[labs1]
X2 = X.loc[labs2]

X1.shape, X2.shape

In [ ]:
X1.iloc[:5, :10]

In [ ]:
X1.mean(axis=0)

In [ ]:
m1, m2 = X1.mean(axis=0), X2.mean(axis=0)      # Series, one value per gene
v1, v2 = X1.var(axis=0, ddof=1), X2.var(axis=0, ddof=1)
n1, n2 = len(X1), len(X2)

In [ ]:
from scipy import stats

se    = np.sqrt(v1/n1 + v2/n2)
tw    = (m1 - m2) / se
dfree = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
# Survival function (also defined as 1 - cdf, but sf is sometimes more accurate).
p = 2 * stats.t.sf(np.abs(tw), dfree)
p

In [ ]:
pd.DataFrame({
    "p": p,
}).describe()

In [ ]:
dft = mc.signature_table(X, labels=labels,  min_abs_lfc=0.6, sort_by='fdr_nominal')

fdr_cutoff = 0.05
lfc_cutoff = 1

dft_all = dft[(dft.fdr_nominal < fdr_cutoff) & (dft.lfc.abs() >= lfc_cutoff)]
print(dft_all.shape)

dft_all.head(2)

In [ ]:
p = dft_all.loc[dft_all.cluster == 1, "p_nominal"]
p.describe()

In [ ]:
p.hist()

### Review clusters - detailed

In [ ]:
small_out = labels[labels == 1].index
X2, d2 = mc.prepare_malignant_matrix(
    keep_genes=mc.program1_panel, drop_pattern=r"^N-",
    keep_samples=[s for s in mc.ms.Z.index if s not in set(small_out)])
cc2 = mc.consensus_cluster(X2)
k2  = mc.choose_k(cc2)
print(k2) 
cc2[k2]["labels"].value_counts().to_dict()

In [ ]:
mc.cluster_summary(cc2)

If min_frac is tiny at every k, don't pick a k. Find the QC axis instead:

In [ ]:
d2["sample_total_Z"].sort_values().head(15)
mc.ms.theta_mal[X2.index].sort_values().head(15)
ties = X2.round(6).apply(lambda c: c.duplicated(keep=False)).mean(axis=1)
ties.sort_values(ascending=False).head(15)

### Critic

In your notebook, cell 72's output showed 'n': 6 and 'n': 119 - the n field cluster_signatures records for each cluster. 

So choose_k picked k=2, but the two groups were 6 samples and 119 samples, not two comparable halves.

In [ ]:
sig[1]['n'], sig[2]['n']

### Each signature

In [ ]:

clu=2
sig[clu].keys()

In [ ]:
sig[clu]['n'], len(sig[clu]['up']), len(sig[clu]['down'])

In [ ]:
ups = set(sig[clu]['up'])
dwns = set(sig[clu]['down'])

ups.intersection(dwns), len(ups.union(dwns))

In [ ]:
clu=1
sig[clu].keys()

In [ ]:
sig[clu]['n'], len(sig[clu]['up']), len(sig[clu]['down'])

In [ ]:
ups = set(sig[clu]['up'])
dwns = set(sig[clu]['down'])

ups.intersection(dwns), len(ups.union(dwns))

In [ ]:
sig[clu]['up']

### signature_table()

In [ ]:
len(labels), labels

In [ ]:
dft = mc.signature_table(X, labels=labels,  min_abs_lfc=0.6, sort_by='fdr_nominal')
dft['abs_lfc'] = dft['lfc'].abs()

fdr_cutoff = 0.05
lfc_cutoff = 1

dft = dft[(dft.fdr_nominal < fdr_cutoff) & (dft.abs_lfc >= lfc_cutoff)]
dft = dft.sort_values('fdr_nominal', ascending=True)

dft.shape

In [ ]:
dft.fdr_nominal.hist()

In [ ]:
dft.head(6)

In [ ]:
genes_sel = sig[clu]['up']

df1 = dft[dft.gene.isin(genes_sel)]
print(f"Number of upregulated genes in cluster {clu}: {df1.shape[0]}")
df1

In [ ]:
genes_clu = np.unique(df1.gene)
print(len(genes_clu))


In [ ]:
genes_clu_in = [x for x in genes_clu if x in df_pivot.index]
df2 = df_pivot.loc[genes_clu_in]

print(df2.shape)
df2.T

### WTCS (Weighted Connectivity Score) and NCS (Normalized Connectivity Score)

WTCS (Weighted Connectivity Score) and NCS (Normalized Connectivity Score) are core metrics used in the LINCS and Connectivity Map (CMap) pipelines to compare query gene signatures against reference expression profiles. 

- WTCS measures signature similarity from −1 to 1
- NCS normalizes these scores within specific cell lines and perturbagen types.

In [ ]:
cond

In [ ]:
cond.index.is_unique 

In [ ]:
cond = cond[~cond.index.duplicated()].loc[df_pivot.columns]
print(cond.index.is_unique)
cond.head(3)

In [ ]:
R1 = mc.score_clusters_vs_tahoe(sig, df_pivot, cond)
R1

In [ ]:
len(R1.targets.unique()), R1.targets.unique()[:20]

In [ ]:
target_list = R1.targets.unique()
len(target_list), len(genes_clu)

In [ ]:
[x for x in target_list if x in genes_clu]

In [ ]:
R1a = R1[ (R1.cluster == clu) & (R1.targets.isin(genes_clu)) & (R1.wtcs.abs() > 0.1) ]
print(R1a.shape)
R1a

In [ ]:
R1a.targets.unique()

In [ ]:
R1a.groupby("cluster").head(10)[["cluster","cell_name","drug","moa-fine","ncs"]]   # reversers

In [ ]:
R1a.groupby("cluster").tail(10)[["cluster","cell_name","drug","moa-fine","ncs"]] 

In [ ]:
out = mc.run(ks=range(2, 9), drop_pattern=r"^N-", min_share=0.3)     # tune from diagnose_filters()
R   = mc.score_clusters_vs_tahoe(out["signatures"], df_pivot, cond)

In [ ]:
R.groupby("cluster").head(10)[["cluster","cell_name","drug","moa-fine","ncs"]]   # reversers

In [ ]:
R.groupby("cluster").tail(10)[["cluster","cell_name","drug","moa-fine","ncs"]] 

In [ ]:
# mc.root_mprog_tahoe = create_dir(mc.root_mprog_cluster / "tahoe")
mc.root_mprog_tahoe

In [ ]:
d = mc.root_mprog_tahoe / "metadata" / "pseudobulk_differential_expression"
files = sorted(d.glob("*.parquet"))
len(files), sum(f.stat().st_size for f in files) / 1e9

In [ ]:
cl = pd.read_parquet(mc.root_mprog_tahoe / "metadata"/ "cell_line_metadata.parquet")
cl[cl.Organ=="Pancreas"][["Cell_ID_Cellosaur","cell_name"]]

In [ ]:
d = mc.diagnose_filters()

d["Z_looks_like_counts"], d["Z_median_of_medians"]


In [ ]:
d["by_min_counts"]

In [ ]:
d["by_min_share"]

In [ ]:
d["joint_grid"]